# Espacios de color RGB y CIE

**Curso**: INF3841 - Recuperación de Información    
**Profesor**: Juan Manuel Barrios  
**Fecha**: 24 de agosto de 2026

En este ejemplo se muestra cómo calcular distancia entre colores en el espacio RGB y en los espacios CIE LAB y CIE LUV.


### Algunas funciones auxiliares para convertir entre espacios de color

In [ ]:
import numpy
import sys
import cv2
import math


# un color se representa como rgb 8bits y lo convierte a cielab o cieluv
class Color:
    def __init__(self, r, g, b, nombre):
        self.r = r
        self.g = g
        self.b = b
        self.nombre = nombre
        # crear un pixel float (valores entre 0 y 1)
        pixel_bgr = numpy.zeros((1, 1, 3), numpy.float32)
        pixel_bgr[0][0] = (self.b / 255, self.g / 255, self.r / 255)
        # convertir a cie-lab es un array de largo 3 con los valores (l,a,b)
        pixel_lab = cv2.cvtColor(pixel_bgr, cv2.COLOR_BGR2Lab)
        self.lab = pixel_lab[0][0]
        # convertir a cie-luv es un array de largo 3 con los valores (l,u,v)
        pixel_luv = cv2.cvtColor(pixel_bgr, cv2.COLOR_BGR2Luv)
        self.luv = pixel_luv[0][0]
        print(
            "{:>15s} rgb=({:3d}, {:3d}, {:3d}) lab=({:8.3f},{:8.3f},{:8.3f}) luv=({:8.3f},{:8.3f},{:8.3f})".format(
                self.nombre,
                self.r, self.g, self.b,
                self.lab[0], self.lab[1], self.lab[2],
                self.luv[0], self.luv[1], self.luv[2],
            )
        )

    def rgb(self):
        return (self.r, self.g, self.b)

    def bgr(self):
        return (self.b, self.g, self.r)

    def cielab(self):
        return self.lab

    def cieluv(self):
        return self.luv


def cuadricular(imagen, size_cuadros=6, color1=(0, 0, 0), color2=(255, 255, 255)):
    flag_par = True
    for y in range(0, imagen.shape[0], size_cuadros):
        use_color1 = not flag_par
        for x in range(0, imagen.shape[1], size_cuadros):
            tl = (x, y)
            br = (x + size_cuadros, y + size_cuadros)
            col = color1 if use_color1 else color2
            cv2.rectangle(imagen, tl, br, col, -1)
            use_color1 = not use_color1
        flag_par = not flag_par


def agregar_texto(imagen, texto, centro, colorTexto, colorFondo):
    face = cv2.FONT_HERSHEY_PLAIN
    scale = 0.7
    thickness = 1
    ((text_w, text_h), baseline_h) = cv2.getTextSize(texto, face, scale, thickness)
    padding = (2, 2)
    tl = (
        int(centro[0] - text_w / 2 - padding[0]),
        int(centro[1] - text_h / 2 - padding[1]),
    )
    br = (
        int(tl[0] + text_w + 2 * padding[0]),
        int(tl[1] + text_h + baseline_h / 4 + 2 * padding[1]),
    )
    cv2.rectangle(imagen, tl, br, colorFondo, -1)
    pos = (int(centro[0] - text_w / 2), int(centro[1] + text_h / 2))
    cv2.putText(imagen, texto, pos, face, scale, colorTexto, thickness, cv2.LINE_AA)

def distancia_euclidiana(c1, c2):
    # se hace el calculo a mano (se podría usar alguna función de numpy)
    diff1 = c1[0] - c2[0]
    diff2 = c1[1] - c2[1]
    diff3 = c1[2] - c2[2]
    dist = math.sqrt(diff1 * diff1 + diff2 * diff2 + diff3 * diff3)
    return dist


def distancia_rgb(color1, color2):
    return distancia_euclidiana(color1.rgb(), color2.rgb())


def distancia_cielab(color1, color2):
    return distancia_euclidiana(color1.cielab(), color2.cielab())


def distancia_cieluv(color1, color2):
    return distancia_euclidiana(color1.cieluv(), color2.cieluv())


# representa la matriz de distancias entre dos lista de colores
class MatrizDistancias:
    def __init__(self, colores1, colores2):
        self.colores1 = colores1
        self.colores2 = colores2

    def calcular_distancias(self, funcion_distancia):
        numcols1 = len(self.colores1)
        numcols2 = len(self.colores2)
        self.dist_matrix = numpy.zeros((numcols1, numcols2), numpy.float32)
        for i in range(0, numcols1):
            for j in range(0, numcols2):
                self.dist_matrix[i][j] = funcion_distancia(self.colores1[i], self.colores2[j])
        # para un ColorMap se crea una imagen gris y luego se convierte a una escala de colores con la función cv2.applyColorMap()
        self.dist_color = numpy.zeros((numcols1, numcols2, 3), numpy.uint8)
        # convertir distancias a escala de grises
        # distancia=0 vale gris=255 (más parecido), distancia=max vale gris=0 (menos parecido)
        val_max = numpy.max(self.dist_matrix)
        for i in range(0, numcols1):
            for j in range(0, numcols2):
                valor_gris = int((1 - (self.dist_matrix[i][j] / val_max)) * 255)
                self.dist_color[i][j] = (valor_gris, valor_gris, valor_gris)
        # calcular ColorMap, el gris se convierte a un color en escala COLORMAP_TURBO (255=rojo, 0=azul)
        # Ver: https://docs.opencv.org/5.0/main_modules/imgproc_colormap.html#colormaptypes
        self.dist_color = cv2.applyColorMap(self.dist_color, cv2.COLORMAP_TURBO)

    def print_distancias(self, nombre):
        numcols1 = len(self.colores1)
        numcols2 = len(self.colores2)
        simetrica = self.colores1 == self.colores2
        val_max = -1
        val_min = numpy.inf
        for i in range(0, numcols1):
            # si es simetrica solo se compara la mitad superior de distancias
            inicio = i + 1 if simetrica else 0
            for j in range(inicio, numcols2):
                if self.dist_matrix[i][j] > val_max:
                    val_max = self.dist_matrix[i][j]
                if self.dist_matrix[i][j] < val_min:
                    val_min = self.dist_matrix[i][j]
        cerca_max = val_max * 0.95
        cerca_min = val_min * 1.05
        # imprimir valores maximos
        print()
        for i in range(0, numcols1):
            inicio = i + 1 if simetrica else 0
            for j in range(inicio, numcols2):
                if self.dist_matrix[i][j] >= val_max:
                    print(" distancia MAX {}={:.2f} ({}, {})".format(nombre, self.dist_matrix[i][j], self.colores1[i].nombre, self.colores2[j].nombre))
        # se imprimen los que están cerca del máximo
        for i in range(0, numcols1):
            inicio = i + 1 if simetrica else 0
            for j in range(inicio, numcols2):
                if cerca_max < self.dist_matrix[i][j] < val_max:
                    print("     -casi MAX {}={:.2f} ({}, {})".format(nombre, self.dist_matrix[i][j], self.colores1[i].nombre, self.colores2[j].nombre))
        # imprimir valores minimos
        print()
        for i in range(0, numcols1):
            inicio = i + 1 if simetrica else 0
            for j in range(inicio, numcols2):
                if self.dist_matrix[i][j] <= val_min:
                    print(" distancia MIN {}={:.2f} ({}, {})".format(nombre, self.dist_matrix[i][j], self.colores1[i].nombre, self.colores2[j].nombre))
        # se imprimen los que están cerca del máximo
        for i in range(0, numcols1):
            inicio = i + 1 if simetrica else 0
            for j in range(inicio, numcols2):
                if val_min < self.dist_matrix[i][j] < cerca_min:
                    print("     -casi MIN {}={:.2f} ({}, {})".format(nombre, self.dist_matrix[i][j], self.colores1[i].nombre, self.colores2[j].nombre))

    def imagen_color1(self, i, block_size):
        imagen = numpy.zeros((block_size, block_size, 3), numpy.uint8)
        imagen[:] = self.colores1[i].bgr()
        return imagen

    def imagen_color2(self, j, block_size):
        imagen = numpy.zeros((block_size, block_size, 3), numpy.uint8)
        imagen[:] = self.colores2[j].bgr()
        return imagen

    def imagen_celda(self, i, j, block_size):
        imagen = numpy.zeros((block_size, block_size, 3), numpy.uint8)
        imagen[:] = self.dist_color[i][j]
        # se escribe el valor de la distancia en el centro de la celda
        texto = "{:4.1f}".format(self.dist_matrix[i][j])
        posicion_centro = (block_size / 2, block_size / 2)
        agregar_texto(imagen, texto, posicion_centro, colorTexto=(220, 250, 250), colorFondo=(100, 100, 100))
        return imagen

    def crear_imagen_matriz(self, block_size, mitad_matriz):
        numcols1 = len(self.colores1)
        numcols2 = len(self.colores2)
        imagen_height = (numcols1 + 1) * block_size
        imagen_width = (numcols2 + 1) * block_size
        imagen = numpy.zeros((imagen_height, imagen_width, 3), numpy.uint8)
        cuadricular(imagen, color1=(170, 200, 170), color2=(220, 220, 220))
        # primera columna, recuadros de colores 1
        start = block_size
        for i in range(0, numcols1):
            imagen[0:block_size, start:start+block_size] = self.imagen_color1(i, block_size)
            start += block_size
        # primera fila, recuadros de colores 2
        start = block_size
        for j in range(0, numcols2):
            imagen[start:start+block_size, 0:block_size] = self.imagen_color2(j, block_size)
            start += block_size
        # celdas de la matriz de distancias
        for i in range(0, numcols1):
            inicio = i + 1 if mitad_matriz else 0
            for j in range(inicio, numcols2):
                start_x = (i + 1) * block_size
                start_y = (j + 1) * block_size
                imagen[start_x:start_x+block_size, start_y:start_y+block_size] = self.imagen_celda(i, j, block_size)
        # linea separadora horizontal entre colores y matriz
        tl = (0, block_size)
        tr = (imagen.shape[1], block_size)
        cv2.line(imagen, tl, tr, (0, 0, 0), 2)
        # linea separadora vertical entre colores y matriz
        tr = (block_size, 0)
        br = (block_size, imagen.shape[0])
        cv2.line(imagen, tr, br, (0, 0, 0), 2)
        return imagen

print("Usando Python {}.{}.{} con OpenCV {}".format(
        sys.version_info.major,
        sys.version_info.minor,
        sys.version_info.micro,
        cv2.__version__))


# Comparar colores RGB con distancia euclidiana

Se comparan entre sí los colores de los 8 vértices del cubo RGB: rojo, verde, azul, magenta, amarillo, cyan, blanco, negro.

La idea es ver la diferencia entre las distancias en el cubo RGB con las distancias en CIE LAB y CIE LUV.

Se calcula la distancia entre colores y se representa con un colormap. Ver función `cv2.applyColorMap()` https://docs.opencv.org/5.0/main_modules/imgproc_colormap.html#applycolormap


In [ ]:
def comparar_colores(matriz_colores, block_size, mitad_matriz):
    # calcular distancias RGB
    matriz_colores.calcular_distancias(distancia_rgb)
    matriz_colores.print_distancias("RGB")
    imagen_rgb = matriz_colores.crear_imagen_matriz(block_size, mitad_matriz)
    # calcular distancias CIE LAB
    matriz_colores.calcular_distancias(distancia_cielab)
    matriz_colores.print_distancias("CIELAB")
    imagen_lab = matriz_colores.crear_imagen_matriz(block_size, mitad_matriz)
    # calcular distancias CIE LUV
    matriz_colores.calcular_distancias(distancia_cieluv)
    matriz_colores.print_distancias("CIELUV")
    imagen_luv = matriz_colores.crear_imagen_matriz(block_size, mitad_matriz)
    # mostrar imagenes
    cv2.imshow("DISTANCIAS_RGB", imagen_rgb)
    cv2.imshow("DISTANCIAS_CIELAB", imagen_lab)
    cv2.imshow("DISTANCIAS_CIELUV", imagen_luv)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


def ejemplo_1():
    # los 8 vértices del cubo rgb
    colores = list()
    colores.append(Color(0, 0, 0, "negro"))
    colores.append(Color(0, 0, 255, "azul"))
    colores.append(Color(0, 255, 0, "verde"))
    colores.append(Color(255, 0, 0, "rojo"))
    colores.append(Color(0, 255, 255, "cyan"))
    colores.append(Color(255, 0, 255, "magenta"))
    colores.append(Color(255, 255, 0, "amarillo"))
    colores.append(Color(255, 255, 255, "blanco"))
    matriz_colores = MatrizDistancias(colores, colores)
    comparar_colores(matriz_colores, block_size=60, mitad_matriz=True)


ejemplo_1()

# Comparación otros colores

Notar como cambian las distancias al comparar los colores en los espacios RGB, CIE LAB y CIE LUV.

A continuación se comparar colores parecidos con LAB y LUV para ver cuál espacio se asemeja más a la opinión de una persona.


In [ ]:
# tomando nombres de colores desde https://www.wikilengua.org/index.php/Lista_de_colores
def ejemplo_2():
    colores1 = list()
    colores1.append(Color(107, 142, 35, "Verde Oliva"))
    colores1.append(Color(153, 102, 204, "Amatista"))
    colores1.append(Color(18, 10, 143, "Azul Marino"))
    colores1.append(Color(237, 145, 33, "Zanahoria"))
    colores2 = list()
    colores2.append(Color(80, 200, 120, "Esmeralda"))
    colores2.append(Color(102, 0, 153, "Púrpura"))
    colores2.append(Color(1, 70, 99, "Azul Petróleo"))
    colores2.append(Color(245, 222, 179, "Beige"))
    # se comparan los dos grupos de colores
    matriz_colores = MatrizDistancias(colores1, colores2)
    comparar_colores(matriz_colores, block_size=90, mitad_matriz=False)


# comparacion entre algunos colores
ejemplo_2()

**Pregunta:**
  - ¿Cuál espacio de los dos espacios (CIE LAB o CIE LUV) se parece más a lo que diría una persona al comparar colores?